In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [47]:
# Clonamos el repositorio original de LaMa
!git clone https://github.com/advimman/lama.git
%cd lama
%env TORCH_HOME=/kaggle/working/lama
%env PYTHONPATH=/kaggle/working/lama

# Instalamos las librerías base
!pip install pytorch-lightning==1.2.10 omegaconf hydra-core==1.1.0 albumentations==0.5.2 webdataset kornia==0.5.0 wget --quiet
!pip install easydict==1.9.0 scikit-image numpy==1.26.4 --quiet

# Limpieza del archivo de requerimientos para evitar conflictos
!sed -i '/scikit-image==0.17.2/d' requirements.txt
!sed -i '/scikit-learn==0.24.2/d' requirements.txt
!pip install -r requirements.txt --no-deps --quiet

Cloning into 'lama'...
remote: Enumerating objects: 478, done.
remote: Counting objects: 100% (323/323), done.
remote: Compressing objects: 100% (195/195), done.
remote: Total 478 (delta 176), reused 128 (delta 128), pack-reused 155 (from 1)
Receiving objects: 100% (478/478), 8.84 MiB | 26.38 MiB/s, done.
Resolving deltas: 100% (194/194), done.
/kaggle/working/lama
env: TORCH_HOME=/kaggle/working/lama
env: PYTHONPATH=/kaggle/working/lama
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 80.9 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incom

In [48]:
!curl -LJO https://huggingface.co/smartywu/big-lama/resolve/main/big-lama.zip
!unzip big-lama.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1056  100  1056    0     0   6683      0 --:--:-- --:--:-- --:--:--  6726
100  363M  100  363M    0     0  29.1M      0  0:00:12  0:00:12 --:--:-- 29.9M
Archive:  big-lama.zip
  inflating: big-lama/config.yaml    
  inflating: big-lama/models/best.ckpt  


In [9]:
import os
print(os.listdir("/kaggle/input/datasets/denisehj"))

['static-vehicles']


In [10]:
!pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [ ]:
import os
os.makedirs('/kaggle/working', exist_ok=True)
os.chdir('/kaggle/working')

In [16]:
%%writefile lama_cleaner.py
import os
import json
import cv2
import io
import numpy as np
from tqdm import tqdm
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseUpload

# === CONFIGURACIÓN DE PRUEBA ==
MODO_PRUEBA = False          # Cambia a False cuando quieras procesar TODO el dataset
MAX_TEST_IMAGES = 2         # Número de imágenes a procesar en la prueba

# === CONFIGURACIÓN DE RUTAS ===
JSON_PATH = '/kaggle/input/datasets/denisehj/static-vehicles/static_vehicles.json' 
DATASET_BASE_DIR = '/kaggle/input/datasets/alvaroquispeunsa/mtc-challenge/train-001'
TOKEN_PATH = '/kaggle/input/datasets/denisehj/mi-token-drive/token.json' 
CARPETA_DESTINO_DRIVE_ID = '1wBWQKNEf8skAHIU_e44zl6OhNffPGGmJ' 

def upload_to_drive(service, filename, image_np, parent_id):
    """ Codifica en memoria y sube a Google Drive """
    success, encoded_img = cv2.imencode('.png', image_np)
    if not success:
        return None

    flujo_archivo = io.BytesIO(encoded_img.tobytes())
    metadatos = {
        'name': filename,
        'mimeType': 'image/png',
        'parents': [parent_id]
    }
    media = MediaIoBaseUpload(flujo_archivo, mimetype='image/png', resumable=True)
    file_drive = service.files().create(body=metadatos, media_body=media, fields='id').execute()
    return file_drive.get('id')

def load_json_data(json_path):
    with open(json_path, 'r') as f:
        return json.load(f)

def generate_masks_and_upload():
    if not os.path.exists(TOKEN_PATH):
        raise FileNotFoundError(f"No se encontró el token de Google Drive en: {TOKEN_PATH}")
        
    creds = Credentials.from_authorized_user_file(TOKEN_PATH, ['https://www.googleapis.com/auth/drive.file'])
    drive_service = build('drive', 'v3', credentials=creds)

    print("Cargando el archivo JSON completo...")
    data = load_json_data(JSON_PATH)
    
    # Si estamos en modo prueba, recortamos el diccionario para el test
    if MODO_PRUEBA:
        print(f"⚠️ MODO PRUEBA ACTIVO: Solo se procesarán las primeras {MAX_TEST_IMAGES} imágenes.")
        items_to_process = list(data.items())[:MAX_TEST_IMAGES]
    else:
        items_to_process = list(data.items())
        print(f"Total de imágenes detectadas en el JSON: {len(items_to_process)}")

    print("Procesando imágenes desde la carpeta descomprimida...")
    
    processed_count = 0
    for img_name, detections in tqdm(items_to_process, desc="Subiendo a Google Drive"):
        
        possible_paths = [
            os.path.join(DATASET_BASE_DIR, "train", f"{img_name}.png"),
            os.path.join(DATASET_BASE_DIR, "train", f"{img_name}.jpg"),
            os.path.join(DATASET_BASE_DIR, f"{img_name}.png"),
            os.path.join(DATASET_BASE_DIR, f"{img_name}.jpg")
        ]
        
        target_image_path = None
        for path in possible_paths:
            if os.path.exists(path):
                target_image_path = path
                break

        if not target_image_path:
            print(f" No se encontró la imagen local para {img_name}, saltando...")
            continue 

        img = cv2.imread(target_image_path)
        if img is None:
            continue

        # Crear la máscara en negro
        h_orig, w_orig = img.shape[:2]
        mask = np.zeros((h_orig, w_orig), dtype=np.uint8)

        # Dibujar detecciones
        for obj in detections:
            cx, cy = obj['cx'], obj['cy']
            w, h = obj['w'], obj['h']
            angle = obj['angle']

            w_dilated = w + 20
            h_dilated = h + 20

            rect = ((cx, cy), (w_dilated, h_dilated), angle)
            box = cv2.boxPoints(rect).astype(int)
            cv2.drawContours(mask, [box], 0, 255, -1)

        # === SUBIDA DIRECTA A GOOGLE DRIVE ===
        #print(f" Subiendo {img_name} y su máscara...")
        upload_to_drive(drive_service, f"{img_name}.png", img, CARPETA_DESTINO_DRIVE_ID)
        upload_to_drive(drive_service, f"{img_name}_mask001.png", mask, CARPETA_DESTINO_DRIVE_ID)
        
        processed_count += 1

    print(f"\n¡Prueba completada! Se procesaron {processed_count} imágenes exitosamente.")

if __name__ == '__main__':
    generate_masks_and_upload()

Overwriting lama_cleaner.py


In [39]:
%%writefile lama_cleaner.py
import os
import json
import cv2
import zipfile
import io
import gc  # Garbage Collector para liberar la memoria RAM de manera forzada
import numpy as np
from tqdm import tqdm
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseUpload

# === CONFIGURACIÓN DE RUTAS ===
JSON_PATH = '/kaggle/input/datasets/denisehj/static-vehicles/static_vehicles.json' 
DATASET_BASE_DIR = '/kaggle/input/datasets/alvaroquispeunsa/mtc-challenge/train-001'
TOKEN_PATH = '/kaggle/input/datasets/denisehj/mi-token-drive/token.json' 

# ID de tu carpeta destino en Google Drive
CARPETA_DESTINO_DRIVE_ID = '10_dPcUVkpvYdP6cbW-mUL0Riva3fLlZS' 

def upload_zip_to_drive(service, filename, zip_bytes, parent_id):
    """ Sube un bloque ZIP en memoria directamente a Google Drive """
    flujo_archivo = io.BytesIO(zip_bytes)
    
    metadatos = {
        'name': filename,
        'mimeType': 'application/zip',
        'parents': [parent_id]
    }
    
    print(f"\n🚀 Subiendo '{filename}' a Google Drive...")
    media = MediaIoBaseUpload(flujo_archivo, mimetype='application/zip', resumable=True)
    file_drive = service.files().create(
        body=metadatos,
        media_body=media,
        fields='id'
    ).execute()
    
    print(f"✓ ¡Subido con éxito! ID en Drive: {file_drive.get('id')}\n")

def load_json_data(json_path):
    with open(json_path, 'r') as f:
        return json.load(f)

def generate_masks_in_chunks():
    if not os.path.exists(TOKEN_PATH):
        raise FileNotFoundError(f"No se encontró el token de Google Drive en: {TOKEN_PATH}")
        
    creds = Credentials.from_authorized_user_file(TOKEN_PATH, ['https://www.googleapis.com/auth/drive.file'])
    drive_service = build('drive', 'v3', credentials=creds)

    print("Cargando el archivo JSON completo...")
    data = load_json_data(JSON_PATH)
    all_items = list(data.items())
    total_imagenes = len(all_items)
    print(f"Total de imágenes detectadas en el JSON: {total_imagenes}")

    # Definimos la división en 4 partes exactas
    NUM_CHUNKS = 4
    chunk_size = int(np.ceil(total_imagenes / NUM_CHUNKS))

    for chunk_idx in range(NUM_CHUNKS):
        inicio = chunk_idx * chunk_size
        fin = min(inicio + chunk_size, total_imagenes)
        
        # Selección del segmento correspondiente al bloque actual
        items_to_process = all_items[inicio:fin]
        nombre_zip = f"dataset_lama_parte_{chunk_idx + 1}_de_4.zip"
        
        print(f"📦 === PROCESANDO BLOQUE {chunk_idx + 1}/{NUM_CHUNKS} ===")
        print(f"Rango de imágenes del JSON: {inicio} al {fin} (Total en este bloque: {len(items_to_process)})")
        
        # Crear un buffer en memoria RAM para este bloque ZIP específico
        buffer_memoria = io.BytesIO()
        processed_count = 0

        with zipfile.ZipFile(buffer_memoria, 'w', zipfile.ZIP_DEFLATED) as out_z:
            for img_name, detections in tqdm(items_to_process, desc=f"Comprimiendo Parte {chunk_idx + 1}"):
                
                possible_paths = [
                    os.path.join(DATASET_BASE_DIR, "train", f"{img_name}.png"),
                    os.path.join(DATASET_BASE_DIR, "train", f"{img_name}.jpg"),
                    os.path.join(DATASET_BASE_DIR, f"{img_name}.png"),
                    os.path.join(DATASET_BASE_DIR, f"{img_name}.jpg")
                ]
                
                target_image_path = None
                for path in possible_paths:
                    if os.path.exists(path):
                        target_image_path = path
                        break

                if not target_image_path:
                    continue 

                img = cv2.imread(target_image_path)
                if img is None:
                    continue

                # Crear la máscara en negro
                h_orig, w_orig = img.shape[:2]
                mask = np.zeros((h_orig, w_orig), dtype=np.uint8)

                # Dibujar las detecciones sobre la máscara
                for obj in detections:
                    cx, cy = obj['cx'], obj['cy']
                    w, h = obj['w'], obj['h']
                    angle = obj['angle']

                    w_dilated = w + 20
                    h_dilated = h + 20

                    rect = ((cx, cy), (w_dilated, h_dilated), angle)
                    box = cv2.boxPoints(rect).astype(int)
                    cv2.drawContours(mask, [box], 0, 255, -1)

                # === ENCODEAR EN JPG (Compresión veloz en memoria) ===
                # 1. Imagen original
                success_img, enc_img = cv2.imencode('.jpg', img, [int(cv2.IMWRITE_JPEG_QUALITY), 95])
                if success_img:
                    out_z.writestr(f"inputs_lama/{img_name}.jpg", enc_img.tobytes())

                # 2. Máscara usando formato de LaMa
                success_mask, enc_mask = cv2.imencode('.jpg', mask, [int(cv2.IMWRITE_JPEG_QUALITY), 95])
                if success_mask:
                    out_z.writestr(f"inputs_lama/{img_name}_mask001.jpg", enc_mask.tobytes())
                    
                processed_count += 1

        # Extraer los bytes del ZIP y transferirlos directamente a Drive
        zip_final_bytes = buffer_memoria.getvalue()
        upload_zip_to_drive(drive_service, nombre_zip, zip_final_bytes, CARPETA_DESTINO_DRIVE_ID)
        
        # 🧹 LIMPIEZA ABSOLUTA DE RAM ANTES DEL SIGUIENTE BLOQUE
        del zip_final_bytes
        buffer_memoria.close()
        del buffer_memoria
        gc.collect() # Forzar a Python a liberar la memoria RAM vaciada
        print(f"✓ Bloque {chunk_idx + 1} completado y liberado de la memoria RAM.\n")

    print("🎉 ¡Proceso maestro finalizado! Los 4 bloques ZIP están a salvo en tu Google Drive.")

if __name__ == '__main__':
    # Modificado para que llame correctamente a la función estructurada en bloques
    generate_masks_in_chunks()

Overwriting lama_cleaner.py


In [40]:
!python lama_cleaner.py

Cargando el archivo JSON completo...
Total de imágenes detectadas en el JSON: 43310
📦 === PROCESANDO BLOQUE 1/4 ===
Rango de imágenes del JSON: 0 al 10828 (Total en este bloque: 10828)
Comprimiendo Parte 1: 100%|███████████████| 10828/10828 [13:37<00:00, 13.24it/s]

🚀 Subiendo 'dataset_lama_parte_1_de_4.zip' a Google Drive...
✓ ¡Subido con éxito! ID en Drive: 17n3m7D0eWG59W3fzJfzHAPwMxLJnyNKE

✓ Bloque 1 completado y liberado de la memoria RAM.

📦 === PROCESANDO BLOQUE 2/4 ===
Rango de imágenes del JSON: 10828 al 21656 (Total en este bloque: 10828)
Comprimiendo Parte 2: 100%|███████████████| 10828/10828 [13:43<00:00, 13.15it/s]

🚀 Subiendo 'dataset_lama_parte_2_de_4.zip' a Google Drive...
✓ ¡Subido con éxito! ID en Drive: 1_QlalxHpOYjMU67fPJpfipc3dGDk7kZw

✓ Bloque 2 completado y liberado de la memoria RAM.

📦 === PROCESANDO BLOQUE 3/4 ===
Rango de imágenes del JSON: 21656 al 32484 (Total en este bloque: 10828)
Comprimiendo Parte 3: 100%|███████████████| 10828/10828 [13:36<00:00, 13.26

In [33]:
import os
import glob

# Aseguramos que estamos fuera de la carpeta por si acaso
os.chdir('/kaggle')

# Buscamos absolutamente todo lo que esté dentro de working
archivos_viejos = glob.glob('/kaggle/working/*') + glob.glob('/kaggle/working/.*')

for f in archivos_viejos:
    try:
        if os.path.isfile(f) or os.path.islink(f):
            os.unlink(f) # Borra archivos o el archivo ZIP gigante corrupto
        elif os.path.isdir(f):
            import shutil
            shutil.rmtree(f) # Borra subcarpetas de imágenes viejas si existen
    except Exception as e:
        print(f"No se pudo borrar {f}: {e}")

# Volvemos a posicionarnos adentro de forma segura
os.chdir('/kaggle/working')
print("¡Limpieza alternativa completada! Todo el espacio ocupado ha sido liberado.")

¡Limpieza alternativa completada! Todo el espacio ocupado ha sido liberado.


In [52]:
#corregir error 
!pip install -U hydra-core

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.0 MB/s eta 0:00:00
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144590 sha256=fbdb4597b7838d3ac8592c4a673814fdeef4661c4a3cd7df841463827eab782d
  Stored in directory: /root/.cache/pip/wheels/1f/be/48/13754633f1d08d1fbfc60d5e80ae1e5d7329500477685286cd
Successfully built antlr4-python3-runtime
  Attempting uninstall: antlr4-python3-runtime
    Found existing installation: antlr4-python3-runtime 4.8
    Uninstalling antlr4-python3-runtime-4.8:
      Successfully uninstalled antlr4-python3-runtime-4.8
  Attempting uninstall: omegaconf
    Found existing installation: omegaconf 2.1.2
    Uninstalling omegaconf-2.1.2:
      Successfully uninstalled omegaconf-2.1.2
  Attempting 

In [83]:
%%writefile test_lama_pipeline_auto.py
import os
import sys
import zipfile
import cv2
import numpy as np
import shutil
import gc
import time
import psutil
import warnings
import dataclasses
import importlib.machinery

warnings.filterwarnings("ignore", category=FutureWarning)

# =====================================================================
# 🔥 PARCHES DE COMPATIBILIDAD E INYECCIÓN DE RENDIMIENTO EXTREMO
# =====================================================================

orig_get_field = dataclasses._get_field
def safe_get_field(cls, name, type, kw_only):
    try:
        return orig_get_field(cls, name, type, kw_only)
    except ValueError as e:
        if "mutable default" in str(e) and cls.__module__ and "hydra" in cls.__module__:
            from dataclasses import Field, MISSING
            f = Field(default=MISSING, default_factory=type, init=True, repr=True,
                      hash=None, compare=True, metadata=None, kw_only=False)
            f.name = name
            f.type = type
            return f
        raise e
dataclasses._get_field = safe_get_field

def universal_find_module(self, fullname, path=None):
    try:
        spec = self.find_spec(fullname, path)
        return spec.loader if spec else None
    except Exception:
        return None
importlib.machinery.FileFinder.find_module = universal_find_module

import hydra
from omegaconf import OmegaConf

RUTAS_DINAMICAS = {
    "indir": "",
    "outdir": "",
    "model_dir": ""
}

def custom_hydra_main(*hydra_args, **hydra_kwargs):
    def decorator(main_func):
        def wrapper(*args, **kwargs):
            config_path = os.path.join(RUTAS_DINAMICAS["lama_repo"], "configs", "prediction", "default.yaml")
            cfg = OmegaConf.load(config_path)
            
            cfg.indir = RUTAS_DINAMICAS["indir"]
            cfg.outdir = RUTAS_DINAMICAS["outdir"]
            cfg.model = OmegaConf.create({
                "path": RUTAS_DINAMICAS["model_dir"],
                "checkpoint": os.path.join(RUTAS_DINAMICAS["model_dir"], "models", "best.ckpt")
            })
            
            cfg.dataset = OmegaConf.create({
                "img_suffix": ".png",
                "pad_out_to_modulo": 8
            })
            cfg.batch_size = 8 
            return main_func(cfg)
        return wrapper
    return decorator

hydra.main = custom_hydra_main

import torch
orig_torch_load = torch.load
def quick_torch_load(path, map_location=None, **kwargs):
    kwargs['weights_only'] = False
    return orig_torch_load(path, map_location=map_location, **kwargs)
torch.load = quick_torch_load

try:
    from saicinpainting.training.evaluation.utils import InpaintingEvaluatorModelWrapper
    orig_forward = InpaintingEvaluatorModelWrapper.forward
    def custom_forward(self, *args, **kwargs):
        with torch.cuda.amp.autocast():
            return orig_forward(self, *args, **kwargs)
    InpaintingEvaluatorModelWrapper.forward = custom_forward
    print("⚡ Parche FP16 Autocast inyectado correctamente.")
except Exception:
    orig_module_call = torch.nn.Module.__call__
    def custom_module_call(self, *args, **kwargs):
        if self.__class__.__name__ == "InpaintingEvaluatorModelWrapper":
            with torch.cuda.amp.autocast():
                return orig_module_call(self, *args, **kwargs)
        return orig_module_call(self, *args, **kwargs)
    torch.nn.Module.__call__ = custom_module_call
    print("⚡ Parche FP16 inyectado mediante llamada global.")

print("🛡️ Todos los parches de compatibilidad y rendimiento activos.")

# =====================================================================
# PIPELINE MULTI-ZIP CON SUBLOTES DE 2000 ELEMENTOS
# =====================================================================
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload

TOKEN_PATH = '/kaggle/input/datasets/denisehj/mi-token-drive/token.json'
TEST_OUTPUT_DIR = './test_lama_results'

CARPETA_INPUT_LAMA_ID = '10_dPcUVkpvYdP6cbW-mUL0Riva3fLlZS'
CARPETA_DESTINO_RESULTADOS_ID = '1jubmTHhur4y3P-U4NIV7sKBvUVbljC6_'

# 🎯 CONFIGURACIÓN AJUSTADA: 2000 IMÁGENES POR BLOQUE
TAMANO_SUB_LOTE = 2000 
NOMBRE_CONTROL_PROGRESO = 'progreso_sub_lotes_global.txt'

def get_current_ram():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 ** 3)

def download_from_drive(service, file_id, output_path):
    request = service.files().get_media(fileId=file_id)
    with open(output_path, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()

def upload_file_to_drive(service, filepath, filename, parent_folder_id, mime_type='application/zip'):
    query = f"'{parent_folder_id}' in parents and name = '{filename}' and trashed = false"
    existentes = service.files().list(q=query, fields="files(id)").execute().get('files', [])
    
    media = MediaFileUpload(filepath, mimetype=mime_type, resumable=True)
    try:
        if existentes:
            file_id = existentes[0]['id']
            print(f"🔄 Actualizando archivo existente en Drive: {filename}...")
            file = service.files().update(fileId=file_id, media_body=media, fields='id').execute()
        else:
            print(f"📤 Subiendo nuevo archivo a Drive: {filename}...")
            file_metadata = {'name': filename, 'parents': [parent_folder_id]}
            file = service.files().create(body=file_metadata, media_body=media, fields='id').execute()
        return file.get('id')
    except Exception as e:
        print(f"⚠️ Error al interactuar con Drive para {filename}: {e}")
        return None

def descargar_progreso_desde_drive(service, parent_folder_id, local_path):
    query = f"'{parent_folder_id}' in parents and name = '{NOMBRE_CONTROL_PROGRESO}' and trashed = false"
    existentes = service.files().list(q=query, fields="files(id)").execute().get('files', [])
    if existentes:
        print(f"📥 Sincronizando historial de progreso global ({NOMBRE_CONTROL_PROGRESO}) desde Drive...")
        download_from_drive(service, existentes[0]['id'], local_path)
        return True
    return False

def obtener_archivos_zip_de_carpeta(service, folder_id):
    print(f"🔍 Buscando todos los paquetes ZIP en la carpeta origen...")
    query = f"'{folder_id}' in parents and mimeType = 'application/zip' and trashed = false"
    resultados = service.files().list(q=query, fields="files(id, name)", pageSize=50).execute()
    files = resultados.get('files', [])
    files = [f for f in files if 'dataset_lama_parte' in f['name']]
    return sorted(files, key=lambda x: x['name'])

def buscar_predict_script():
    posibles_rutas = ["bin/predict.py", "lama/bin/predict.py"]
    for ruta in posibles_rutas:
        if os.path.exists(ruta): return os.path.abspath(ruta)
    for root, dirs, files in os.walk('.'):
        if 'predict.py' in files and 'bin' in root:
            return os.path.abspath(os.path.join(root, 'predict.py'))
    return None

def main():
    if not os.path.exists(TOKEN_PATH):
        raise FileNotFoundError(f"Falta el token de Google Drive en: {TOKEN_PATH}")
        
    creds = Credentials.from_authorized_user_file(TOKEN_PATH, ['https://www.googleapis.com/auth/drive.file'])
    drive_service = build('drive', 'v3', credentials=creds)

    predict_script = buscar_predict_script()
    if not predict_script:
        print("❌ Error: No se encontró 'bin/predict.py'.")
        return
        
    lama_repo_dir = os.path.dirname(os.path.dirname(predict_script))
    model_path = os.path.join(lama_repo_dir, "big-lama")
    if not os.path.exists(model_path):
        model_path = os.path.abspath("big-lama")

    archivos_drive = obtener_archivos_zip_de_carpeta(drive_service, CARPETA_INPUT_LAMA_ID)
    if not archivos_drive:
        print("❌ No se encontraron archivos ZIP en la carpeta origen de Drive.")
        return

    print(f"📋 Se detectaron {len(archivos_drive)} paquetes ZIP para procesar.")

    import runpy
    sys.path.insert(0, lama_repo_dir)
    os.environ["PYTHONPATH"] = lama_repo_dir

    archivo_progreso_local = NOMBRE_CONTROL_PROGRESO
    sub_lotes_completados = set()
    if descargar_progreso_desde_drive(drive_service, CARPETA_DESTINO_RESULTADOS_ID, archivo_progreso_local):
        with open(archivo_progreso_local, "r") as f:
            for line in f:
                line = line.strip()
                if line: sub_lotes_completados.add(line)

    for idx, zip_drive_info in enumerate(archivos_drive):
        nombre_zip_remoto = zip_drive_info['name']
        nombre_sin_ext, _ = os.path.splitext(nombre_zip_remoto)

        print("\n" + "="*70)
        print(f"📦 [ZIP {idx + 1}/{len(archivos_drive)}] PROCESANDO PAQUETE: {nombre_zip_remoto}")
        print("="*70)
        
        local_zip = "lote_actual_entrada.zip"
        print(f"📥 Descargando archivo ZIP desde Google Drive...")
        download_from_drive(drive_service, zip_drive_info['id'], local_zip)

        print("🔍 Escaneando archivos e índices de pares e imágenes...")
        todos_los_pares = []
        with zipfile.ZipFile(local_zip, 'r') as zip_ref:
            lista_archivos = [f for f in zip_ref.namelist() if not f.endswith('/')]
            imagenes_base = [f for f in lista_archivos if "_mask" not in f and f.lower().endswith(('.jpg', '.png', '.jpeg'))]
            
            for img_path in imagenes_base:
                n_base = os.path.basename(img_path)
                n_sin_ext, _ = os.path.splitext(n_base)
                
                mascara_path = None
                for f in lista_archivos:
                    if f"{n_sin_ext}_mask" in os.path.basename(f):
                        mascara_path = f
                        break
                if mascara_path:
                    todos_los_pares.append((img_path, mascara_path, n_sin_ext))

        total_pares = len(todos_los_pares)
        print(f"📊 Total mapeado: {total_pares} pares. Sub-lotes de {TAMANO_SUB_LOTE} elementos.")

        for i in range(0, total_pares, TAMANO_SUB_LOTE):
            indice_sub_lote = i // TAMANO_SUB_LOTE
            numero_sub_lote_legible = indice_sub_lote + 1
            
            id_unico_bloque = f"{nombre_sin_ext}_sublote_{numero_sub_lote_legible}"

            if id_unico_bloque in sub_lotes_completados:
                print(f"⏭️ Saltando [{nombre_sin_ext} - Sub-lote {numero_sub_lote_legible}]. Ya fue procesado.")
                continue

            sub_lote = todos_los_pares[i:i + TAMANO_SUB_LOTE]
            print(f"\n   -> 🔄 Procesando Sub-lote [{numero_sub_lote_legible}]: Elementos del {i} al {min(i + TAMANO_SUB_LOTE, total_pares)}")
            
            temp_input_dir = os.path.abspath("./sub_inputs_temp")
            temp_output_dir = os.path.abspath("./sub_outputs_temp")
            os.makedirs(temp_input_dir, exist_ok=True)
            os.makedirs(temp_output_dir, exist_ok=True)

            with zipfile.ZipFile(local_zip, 'r') as zip_ref:
                for img_path, mask_path, name in sub_lote:
                    img_data = zip_ref.read(img_path)
                    img_np = np.frombuffer(img_data, np.uint8)
                    img_cv = cv2.imdecode(img_np, cv2.IMREAD_COLOR)
                    if img_cv is not None:
                        img_resized = cv2.resize(img_cv, (640, 640), interpolation=cv2.INTER_AREA)
                        cv2.imwrite(os.path.join(temp_input_dir, f"{name}.png"), img_resized)

                    mask_data = zip_ref.read(mask_path)
                    mask_np = np.frombuffer(mask_data, np.uint8)
                    mask_cv = cv2.imdecode(mask_np, cv2.IMREAD_GRAYSCALE)
                    if mask_cv is not None:
                        mask_resized = cv2.resize(mask_cv, (640, 640), interpolation=cv2.INTER_NEAREST)
                        cv2.imwrite(os.path.join(temp_input_dir, f"{name}_mask001.png"), mask_resized)

            RUTAS_DINAMICAS["lama_repo"] = lama_repo_dir
            RUTAS_DINAMICAS["indir"] = temp_input_dir
            RUTAS_DINAMICAS["outdir"] = temp_output_dir
            RUTAS_DINAMICAS["model_dir"] = model_path

            sys.argv = [predict_script]
            runpy.run_path(predict_script, run_name='__main__')

            # 🏷️ NUEVO FORMATO DE NOMBRE DE SALIDA
            nombre_zip_parcial = f"resultado_{nombre_sin_ext}_sublote_{numero_sub_lote_legible}.zip"
            path_zip_parcial = os.path.join(os.getcwd(), nombre_zip_parcial)
            
            archivos_salida = os.listdir(temp_output_dir) if os.path.exists(temp_output_dir) else []
            if archivos_salida:
                with zipfile.ZipFile(path_zip_parcial, 'w', zipfile.ZIP_DEFLATED) as zipf:
                    for f_salida in archivos_salida:
                        if f_salida.endswith('.png'):
                            zipf.write(os.path.join(temp_output_dir, f_salida), f_salida)
                
                upload_file_to_drive(drive_service, path_zip_parcial, nombre_zip_parcial, CARPETA_DESTINO_RESULTADOS_ID)
                
                with open(archivo_progreso_local, "a") as f_prog:
                    f_prog.write(f"{id_unico_bloque}\n")
                sub_lotes_completados.add(id_unico_bloque)
                upload_file_to_drive(drive_service, archivo_progreso_local, NOMBRE_CONTROL_PROGRESO, CARPETA_DESTINO_RESULTADOS_ID, mime_type='text/plain')

            if os.path.exists(path_zip_parcial): os.remove(path_zip_parcial)
            shutil.rmtree(temp_input_dir, ignore_errors=True)
            shutil.rmtree(temp_output_dir, ignore_errors=True)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()
            print(f"✨ Sub-lote {numero_sub_lote_legible} completado con éxito y subido a Drive.")

        if os.path.exists(local_zip): os.remove(local_zip)
        print(f"✅ Finalizado el paquete entero: {nombre_zip_remoto}")

    print("\n🎉 [PROCESO AUTOMÁTICO COMPLETO PARA TODOS LOS ZIP FINALIZADO]")

if __name__ == '__main__':
    main()

Overwriting test_lama_pipeline_auto.py


In [ ]:
!python test_lama_pipeline_auto.py

⚡ Parche FP16 inyectado mediante llamada global de módulos.
🛡️ Todos los parches de compatibilidad y rendimiento activos.
🔍 Buscando archivos ZIP en la carpeta origen de Drive ID: 10_dPcUVkpvYdP6cbW-mUL0Riva3fLlZS...
📋 Se detectaron 4 archivos ZIP en Drive.
📥 Se encontró un archivo de progreso en Drive. Sincronizando estado...

📦 [ZIP 1/4] PROCESANDO RECURSIVO: dataset_lama_parte_1_de_4.zip
📥 Descargando lote comprimido grande desde Drive...
🔍 Escaneando índice del ZIP...
📊 Total mapeado en este ZIP: 10828 pares. Procesando en bloques de 200...
⏭️ Saltando bloque [1]. Ya fue procesado y subido previamente.
⏭️ Saltando bloque [2]. Ya fue procesado y subido previamente.

   -> 🔄 Trabajando en Sub-lote [3]: Elementos del 400 al 600
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/imports.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-